## IMPORTS

In [ ]:
import warningsimport pandas as pdimport torchfrom torch import nnfrom hazm import Normalizer, word_tokenize, stopwords_listfrom collections import Counterimport stringfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.metrics import precision_score, recall_score, f1_scorefrom sklearn.model_selection import train_test_splitfrom timeit import default_timer as timerimport matplotlib.pyplot as pltfrom tqdm import tqdmimport textwrapwarnings.filterwarnings('ignore')

## Device

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Timer

In [3]:
def print_train_time(start: float, end : float, device: torch.device = None):    total_time = end - start    print(f"Train time on {device}: {total_time/60:.3f} minutes\n\n")    return total_time

## DATA

In [4]:
emails = pd.read_csv("data/emails.csv")

In [5]:
emails.head()

,text,label
0,﻿ممنون آقا سامان.\nمن پارسال اصلا آزاد شرکت نک...,ham
1,﻿سلام آقای کریمی\nبالاخره آزمونارشد تموم شد من...,ham
2,﻿درود بر حاج وحیدی بنده بعنوان یک دکتری تاریخ ...,ham
3,﻿با سلام و احترام\nضمن تقدیر از مسولین محترم ...,ham
4,﻿با سلام اینجانب یک دستگاه خودرو پراید 131 با ...,ham


## Q1: Preprocessing

In [ ]:
# Hazm componentsnormalizer = Normalizer()stopwords = set(stopwords_list())punctuations = string.punctuation + "،؛؟«»"def preprocess_persian_text(text):    # Normalize    text = normalizer.normalize(text).replace('\ufeff', '').replace('\u200c', '')    # Tokenize and clean    tokens = [        ''.join(c for c in token if c not in punctuations)        for token in word_tokenize(text)    ]    # Remove stopwords and empty tokens    return [t for t in tokens if t and t not in stopwords]# Apply preprocessingemails['processed_text_tokens'] = emails['text'].apply(preprocess_persian_text)emails['processed_text_joined'] = emails['processed_text_tokens'].apply(' '.join)for _, row in emails.head().iterrows():    print(f"Original: {row['text']}")    print(f"Processed Tokens: {row['processed_text_tokens']}")    print(f"Processed Joined: {row['processed_text_joined']}\n")

Original: ﻿ممنون آقا سامان.
من پارسال اصلا آزاد شرکت نکرده بودم و سراسری هم قبول نشدم. فقط میخواستم بدونم شرایط چطوریه واسه سال بعد. مرسی از راهنمایی هاتون.
Processed Tokens: ['ممنون', 'آقا', 'سامان', 'پارسال', 'اصلا', 'آزاد', 'شرکت', 'نکردهبودم', 'سراسری', 'قبول', 'نشدم', 'میخواستم', 'بدونم', 'شرایط', 'چطوریه', 'واسه', 'سال', 'مرسی', 'راهنمایی', 'هاتون']
Processed Joined: ممنون آقا سامان پارسال اصلا آزاد شرکت نکردهبودم سراسری قبول نشدم میخواستم بدونم شرایط چطوریه واسه سال مرسی راهنمایی هاتون

Original: ﻿سلام آقای کریمی
بالاخره آزمونارشد تموم شد من راحت شدم
یکم راهنمایی می خوام
بهترین مجله ی تخصصی زبان شناسی که بتونم اشتراک بگیرم کدومه؟
و در زمینه ی جامعه شناسی و روانشناسی زبان کتابای خوب و مرجع رو لطفا بهم معرفی کنید. صرفا به خاطر علاقه ی خودم می خوام مطالعه ی آزاد و تخصصی داشته باشم برای کنکور نیست
مرسی
دوستان دیگه هم اگه کمک کنند ممنونمی شم 
Processed Tokens: ['سلام', 'کریمی', 'بالاخره', 'آزمونارشد', 'تموم', 'راحت', 'شدم', 'یکم', 'راهنمایی', 'میخوام', 'مجلهی', 'تخصصی', 'زبانشناسی', 